# JSON Basics — 06: JSON Performance

JSON is the slowest analytics format. This notebook benchmarks it
against Parquet and Avro, and shows optimization techniques.

Topics: why JSON is slow, compression benefit, JSON vs Parquet benchmark,
converting to Parquet as first pipeline step.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Why JSON is Slow for Analytics



In [ ]:

print("""
Why JSON is slow for analytical queries:
  1. No column pruning — entire line must be parsed even for 1 column
  2. No predicate pushdown — filters applied AFTER parsing
  3. No statistics — cannot skip rows without reading
  4. Text parsing overhead — JSON parsing is CPU-intensive
  5. Large file size — field names repeated on every row

  Row: {"order_id":1,"customer_id":101,"product":"Laptop","category":"Electronics",...}
       ↑ field name repeated on every row — massive waste for 1M rows

  Parquet: field names stored ONCE in footer, data stored as binary
""")


## Step 2 — Benchmark: JSON vs Parquet vs Avro



In [ ]:

import random, datetime, pathlib, json as pyjson

random.seed(42); N=200_000
rows=[{"order_id":i,"customer_id":random.randint(1,10000),
       "category":random.choice(["Electronics","Books","Clothing","Food"]),
       "country":random.choice(["US","UK","DE","FR","JP"]),
       "quantity":random.randint(1,10),"price":round(random.uniform(5,1000),2),
       "revenue":0.0,"status":random.choice(["pending","confirmed","shipped"]),
       "order_date":str(datetime.date(2023,1,1)+datetime.timedelta(days=random.randint(0,365)))}
      for i in range(N)]
for r in rows: r["revenue"]=round(r["quantity"]*r["price"],2)

json_path = f"{DATA_DIR}/bench.json"
with open(json_path,"w") as f:
    for r in rows: f.write(pyjson.dumps(r)+"\n")

df_from_json=spark.read.json(json_path)
PQ_PATH=f"{DATA_DIR}/bench.parquet"
AV_PATH=f"{DATA_DIR}/bench.avro"
df_from_json.write.mode("overwrite").parquet(PQ_PATH)
df_from_json.write.format("avro").mode("overwrite").save(AV_PATH)

json_mb=pathlib.Path(json_path).stat().st_size/1024/1024
pq_mb=sum(pathlib.Path(f).stat().st_size for f in pathlib.Path(PQ_PATH).glob("*.parquet"))/1024/1024
av_mb=sum(pathlib.Path(f).stat().st_size for f in pathlib.Path(AV_PATH).glob("*.avro"))/1024/1024
print(f"Storage ({N:,} rows):")
print(f"  JSON    : {json_mb:.1f} MB")
print(f"  Avro    : {av_mb:.1f} MB  ({json_mb/av_mb:.1f}x smaller)")
print(f"  Parquet : {pq_mb:.1f} MB  ({json_mb/pq_mb:.1f}x smaller)")


## Step 3 — Query Performance Benchmark



In [ ]:

import glob as glib

benchmarks=[
    ("Sum revenue",   lambda p,f: spark.read.format(f).load(p).agg(F.sum("revenue")).collect()),
    ("Filter+count",  lambda p,f: spark.read.format(f).load(p).filter(F.col("category")=="Electronics").count()),
    ("GroupBy",       lambda p,f: spark.read.format(f).load(p).groupBy("category").agg(F.sum("revenue")).collect()),
]
print(f"{'Query':<20} {'JSON':>8} {'Avro':>8} {'Parquet':>10}")
print("-"*50)
for label,fn in benchmarks:
    tj,ta,tp=[],[],[]
    for _ in range(3):
        t0=time.time(); fn(json_path,"json");    tj.append(time.time()-t0)
        t0=time.time(); fn(AV_PATH,"avro");      ta.append(time.time()-t0)
        t0=time.time(); fn(PQ_PATH,"parquet");   tp.append(time.time()-t0)
    t_j,t_a,t_p=sorted(tj)[1],sorted(ta)[1],sorted(tp)[1]
    print(f"  {label:<18} {t_j:>6.3f}s {t_a:>6.3f}s {t_p:>8.3f}s")
print()
print("JSON is the slowest: text parsing + no column pruning + no stats")


## Step 4 — JSON to Parquet as First Step



In [ ]:

def ingest_json_to_parquet(json_input, parquet_output, schema):
    """Production pattern: convert JSON landing zone to Parquet immediately."""
    df = spark.read.schema(schema).json(json_input)
    count = df.count()
    df.write.mode("overwrite").option("compression","zstd").parquet(parquet_output)
    verify = spark.read.parquet(parquet_output).count()
    print(f"  Ingested: {count:,} rows → {verify:,} in Parquet  ({'✅' if count==verify else '❌'})")
    return count

from pyspark.sql.types import *
schema=StructType([StructField(k,t) for k,t in [
    ("order_id",LongType()),("customer_id",LongType()),("category",StringType()),
    ("country",StringType()),("quantity",IntegerType()),("price",DoubleType()),
    ("revenue",DoubleType()),("status",StringType()),("order_date",DateType())]])

ingest_json_to_parquet(json_path, f"{DATA_DIR}/ingested_parquet", schema)


## Summary



In [ ]:

print("""
JSON performance rules:
  1. NEVER use JSON as your analytics storage format
  2. Always convert JSON to Parquet as the first pipeline step
  3. Use gzip compression for JSON files at rest (saves 3-5x storage)
  4. Always define explicit schema (inferSchema is slow on large JSON)
  5. Use NDJSON (one object per line) — easier to parallelize

JSON is acceptable for:
  - Configuration files
  - REST API payloads (ingestion only)
  - Human-readable small exports
  - Kafka message value format

JSON is NOT acceptable for:
  - Analytics storage
  - Long-term data lake storage
  - Frequently queried tables
  - Files > 100 MB that you query repeatedly
""")
